# מעבדה 09 — היחס היסודי

במעבדה הזו תמסרו למחשב פונקציה אחת, $S(U, V, N)$, ותו לא — ותקבלו בחזרה טמפרטורות, לחצים,
מצבי שיווי משקל וקיבולי חום, על ידי גזירתה ומקסומה. אחר כך תריצו שוב את הסימולציה של מודול 01
ותנהלו את מאזן האנטרופיה שלה.

המסלול: בונים שני יחסים יסודיים וקוראים את השיפועים שלהם (חלק 1); מוצאים היכן שני מוצקים
מפסיקים להחליף אנרגיה (חלק 2); משחררים את הדופן של גז מחולק, אילוץ אחר אילוץ (חלק 3); בודקים
את השיפועים מול צורות סגורות ומודדים כמה מהר השגיאה קטנה (חלק 4); מנהלים את מאזן האנטרופיה של
הסימולציה של מודול 01 (חלקים 5 ו-6); בוחנים את האקסטנסיביות, ושוברים אותה (חלק 7); קוראים
יציבות מתוך העקמומיות (חלק 8); מודדים ייצור אנטרופיה בניסוי ערבוב ממשי (חלק 9); מריצים את
הבדיקות האוטומטיות (חלק 10); וחוקרים בחופשיות (חלק 11).

עברו עליה בסדר. במקום שבו המחברת מבקשת מכם לנבא, רשמו את ניבויכם בתא המיועד לכך **לפני**
הרצת התא הבא. ניבוי שהתחייבתם אליו הוא הדרך האמינה היחידה לגלות שטעיתם.

## מפרט המודל

| | |
|---|---|
| **מערכת** | יחס יסודי $S(U, V, N)$ — $S(U, n)$ של מוצק איינשטיין ו-$S(U, V, N)$ של סאקור–טטרודה עבור הגז האידיאלי החד-אטומי — לבדו או כשתי המחציות של מערכת מורכבת |
| **דינמיקה** | אין עבור המשטח: שחרור אילוץ פירושו מקסום האנטרופיה הכוללת על פני מה שהדופן מאפשרת לנוע; סימולציית קפיצות הקוונטים של מודול 01 מספקת דינמיקה אחת (חלקים 5–6) |
| **גבול** | המערכת המורכבת מבודדת; הדופן הפנימית אדיאבטית או דיאתרמית, קבועה או ניידת, אטומה או מחוררת |
| **צבר** | תרמודינמיקה של שיווי משקל — כל נקודה על המשטח היא מצב שיווי משקל |
| **מוזנח** | פלוקטואציות סביב המקסימום, איברי שטח, כוחות ארוכי טווח |
| **תקף כאשר** | תת-המערכות מאקרוסקופיות; צורת איינשטיין — כאשר קירוב סטירלינג תקף; סאקור–טטרודה — עבור גז קלאסי מדולל |
| **אופני כישלון** | מערכות קטנות (תיקוני $\ln N$), טמפרטורה נמוכה (סאקור–טטרודה הופכת לבלתי פיזיקלית), טלאים קמורים (דו-קיום פאזות, כוחות ארוכי טווח) |

כל הפיזיקה נמצאת ב-`thermolab.fundamental` וב-`thermolab.equilibrium` — פתחו וקראו אותם. דבר
בקורס הזה אינו מוסתר בתוך מסגרת תוכנה.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import engines, equilibrium, fundamental, processes
from thermolab.constants import AMU, K_B, N_A
from thermolab.validation import convergence_study, relative_error, seed_study

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

QUANTUM = 5.0 * K_B  # the Einstein solids' energy quantum: epsilon / k_B = 5 K
ARGON_MASS = 39.948 * AMU

solid = fundamental.einstein_solid(QUANTUM)
argon = fundamental.monatomic_ideal_gas(ARGON_MASS)
print(f"k_B = {K_B:.6e} J/K,  quantum = {QUANTUM:.3e} J")

## חלק 1 — פונקציה אחת, והשיפועים שלה

יחס כאן הוא פשוט פונקציית Python בשם `s(u, v, n)` המחזירה אנטרופיה ביחידות J/K. דבר נוסף אינו
מסופק: לא טמפרטורה, לא לחץ, לא משוואת מצב. קחו מול אחד של ארגון בטמפרטורה
$300\ \mathrm{K}$ ובלחץ $1\ \mathrm{bar}$ — חישוב $U$ ו-$V$ עבורו הוא המקום היחיד שבו אנו
משתמשים במה שכבר ידוע לנו — ובקשו מן הפונקציה את השיפועים שלה.

In [ ]:
n_atoms = N_A
u = 1.5 * n_atoms * K_B * 300.0
v = n_atoms * K_B * 300.0 / 1e5

slopes = fundamental.entropy_slopes(argon, u, v, n_atoms)
print(f"S                  = {float(argon(u, v, n_atoms)):.2f} J/K   "
      f"(tabulated for argon: 154.8 at 298 K)")
print(f"1/T  = dS/dU       -> T  = {slopes.temperature:.6f} K")
print(f"P/T  = dS/dV       -> P  = {slopes.pressure:.4f} Pa")
print(f"-mu/T = dS/dN      -> mu = {slopes.chemical_potential:.4e} J per atom")
print(f"\nPV / (N k_B T)     = {slopes.pressure * v / (n_atoms * K_B * slopes.temperature):.9f}")

### לנבא

התחייבו לתשובה לכל אחד מארבעת הניבויים של דף המודול לפני שתמשיכו — הם מה שהמעבדה הזו בודקת:

1. גוש נחושת חם של 2 kg מול גוש קר של 0.2 kg: אנרגיות שוות, טמפרטורות שוות, את שתיהן, או
   אף אחת מהן?
2. האם האנטרופיה הכוללת של המוצקים של מודול 01, בזמן שהם מגיעים לרלקסציה, יכולה אי פעם
   *לרדת* בצעד בודד?
3. גז מכפיל את נפחו אל תוך ריק. האם האנטרופיה שלו עולה, יורדת או נשארת קבועה?
4. שתי חתיכות זהות של חומר שעקומת $S(U)$ שלו מתעקמת כלפי מעלה, במגע באותה טמפרטורה: מה
   קורה?

**הניבויים שלכם:**

1.
2.
3.
4.

## חלק 2 — היכן שני מוצקים נעצרים

למוצק A יש 40 מתנדים ולמוצק B יש 400; יחד הם מחזיקים $5 \times 440$ קוונטים. הדופן ביניהם
מוליכה חום. סרקו כל דרך לחלוק את האנרגיה, חברו את שתי האנטרופיות, ומצאו את השיא.

In [ ]:
total = 5.0 * 440 * QUANTUM
pair = fundamental.Composite(solid, solid,
                             fundamental.Part(0.5 * total, 1.0, 40.0),
                             fundamental.Part(0.5 * total, 1.0, 400.0))
u_a, s_a, s_b = pair.energy_scan(n_points=2001, margin=0.002)
s_tot = s_a + s_b
peak = int(np.argmax(s_tot))
share = u_a / total

fig, (top, bottom) = plt.subplots(1, 2, figsize=(11, 3.8))
top.plot(share, (s_tot - s_tot.max()) / K_B, color="black")
top.axvline(share[peak], color="grey", ls=":")
top.set_xlim(0, 0.4)
top.set_ylim(-60, 5)
top.set_xlabel("U_A / U")
top.set_ylabel("(S_A + S_B - S_max) / k_B")
top.set_title("total entropy along the partition")

t_a = [fundamental.temperature_of(solid, x, 1.0, 40.0) for x in u_a[::20]]
t_b = [fundamental.temperature_of(solid, total - x, 1.0, 400.0) for x in u_a[::20]]
bottom.plot(share[::20], t_a, color="#dc2626", label="T_A")
bottom.plot(share[::20], t_b, color="#2563eb", label="T_B")
bottom.axvline(share[peak], color="grey", ls=":")
bottom.set_xlim(0, 0.4)
bottom.set_ylim(0, 150)
bottom.set_xlabel("U_A / U")
bottom.set_ylabel("temperature (K)")
bottom.legend()
bottom.set_title("each solid's temperature, from its own slope")
plt.tight_layout()
plt.show()

best = pair.released("energy")
print(f"scanned peak at U_A/U  = {share[peak]:.4f}")
print(f"maximiser: U_A/U       = {best.a.energy / total:.6f}   "
      f"(n_A / (n_A + n_B) = {40 / 440:.6f})")
print(f"T_A = {best.slopes_a().temperature:.4f} K,   T_B = {best.slopes_b().temperature:.4f} K")
print(f"U_B / U_A              = {best.b.energy / best.a.energy:.4f}")

השיא נמצא במקום שבו שתי הטמפרטורות נחתכות, והוא יושב כאשר אחד חלקי אחד-עשר מן האנרגיה נמצא
ב-A. טמפרטורות שוות, ואנרגיות ביחס $1 : 10$ — תמונת האנרגיות השוות של ניבוי 1 פשוט אינה המקום
שבו נמצא המקסימום. שימו לב גם שהממקסם מצא את השיא בלי שנאמר לו דבר על טמפרטורה: כל מה שעשה
אי פעם הוא להשוות אנטרופיות כוללות.

## חלק 3 — שחרור הדופן של גז מחולק

תיבה מחזיקה ארגון המחולק על ידי דופן: בצד A נמצא שליש מן האטומים, דחוסים לתוך $12\%$ מן
הנפח, ובידיו יותר מחלקו באנרגיה. שחררו את האילוצים אחד אחד והתבוננו מה משתווה. אחר כך נסו את
הסדר האחר.

המעבדה מסרבת בכוונה לדבר אחד: לשחרר את הדופן להחליק תוך שהיא נשארת מבודדת. דופן נעה מבצעת
עבודה על שני הצדדים, ולכן אנרגיה עוברת בכל פעם שנפח עובר; נסו את `start.released("volume")`
וקראו את הודעת השגיאה.

In [ ]:
n_a, n_b = 1e20, 2e20
u_total = 1.5 * (n_a + n_b) * K_B * 300.0
v_total = 3e-3
start = fundamental.Composite(argon, argon,
                              fundamental.Part(0.62 * u_total, 0.12 * v_total, n_a),
                              fundamental.Part(0.38 * u_total, 0.88 * v_total, n_b))


def describe(label, c):
    a, b = c.slopes_a(), c.slopes_b()
    print(f"{label:<26} T = {a.temperature:7.2f} | {b.temperature:7.2f} K   "
          f"P = {a.pressure:9.1f} | {b.pressure:9.1f} Pa   "
          f"mu = {a.chemical_potential:.4e} | {b.chemical_potential:.4e} J   "
          f"S_tot - S_0 = {(c.total_entropy - start.total_entropy) / K_B:.4e} k_B")


describe("as prepared", start)
diathermal = start.released("energy")
describe("wall conducts heat", diathermal)
piston = diathermal.released("energy", "volume")
describe("... and slides", piston)
describe("... and is perforated", piston.released("energy", "particles"))
print()
perforated_first = diathermal.released("energy", "particles")
describe("other order: perforated", perforated_first)
describe("... then slides", perforated_first.released("energy", "volume"))

כל שחרור יכול רק להעלות את האנטרופיה הכוללת, וכל אחד משווה בדיוק שיפוע אחד נוסף. ברגע
ש-$T$ ו-$P$ מסכימים, ניקוב הדופן אינו משנה דבר: יחס גיבס–דוהם אינו משאיר ל-$\mu$ חופש להיות
שונה בין שתי דגימות של אותו גז באותה טמפרטורה ובאותו לחץ. בשחרור בסדר האחר — חום תחילה, אחר
כך חורים — הגז מסיים באותו מצב סופי. התבוננו היטב בעמודת האנטרופיה כדי לראות מדוע.

## חלק 4 — שיפועים מול צורות סגורות, וכמה מהר השגיאה קטנה

עבור מוצק איינשטיין יש לטמפרטורה צורה סגורה,
$T = \varepsilon / (\kB \ln(1 + n\varepsilon/U))$. הטמפרטורה של המעבדה היא הפרש מרכזי עם צעד
יחסי $h$. הפרש מרכזי אמור להיות מסדר שני: הקטינו את $h$ לחצי, והשגיאה אמורה לקטון פי ארבעה.

In [ ]:
u, n = 2.0 * 300 * QUANTUM, 300.0
exact = float(fundamental.einstein_solid_temperature(u, n, QUANTUM))
steps = [5, 10, 20, 40, 80]  # h = 1/steps
study = convergence_study(
    lambda k: fundamental.temperature_of(solid, u, 1.0, n, rel_step=1.0 / k), steps, exact)
for k, err in zip(steps, study.errors, strict=True):
    print(f"h = {1 / k:.4f}   relative error = {err:.3e}")
print(f"\nobserved order = {study.observed_order:.3f}   (central difference: 2)")

print("\nhot solid vs module 01's map T = q epsilon / (n k_B):")
for per_oscillator in (1, 10, 100, 1000):
    u_hot = per_oscillator * n * QUANTUM
    t_exact = float(fundamental.einstein_solid_temperature(u_hot, n, QUANTUM))
    t_equipartition = u_hot / (n * K_B)
    print(f"  {per_oscillator:5d} quanta/oscillator: T = {t_exact:10.3f} K,  "
          f"equipartition {t_equipartition:10.3f} K,  offset {t_exact - t_equipartition:.4f} K")

הסדר יוצא 2, וככל שהמוצק מתחמם ההיסט מן המפה של מודול 01 מתייצב על $2.5\ \mathrm{K}$ — מחצית
מ-$\varepsilon/\kB = 5\ \mathrm{K}$. מודול 01 הניח את המפה של חלוקת האנרגיה השווה; הספירה גזרה
אותה, יחד עם התיקון הראשון שלה.

## חלק 5 — מאזן האנטרופיה של הסימולציה של מודול 01

### לנבא

לפני הרצת התא הבא: המוצקים של מודול 01 — 300 ו-100 מתנדים, בטמפרטורות $500\ \mathrm{K}$
ו-$250\ \mathrm{K}$ — מגיעים לרלקסציה קוונט אחר קוונט. אתם תעקבו אחר האנטרופיה הכוללת המדויקת
שלהם אחרי כל צעד. האם היא תרד אי פעם בצעד בודד? היכן היא תסתיים?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
state = equilibrium.from_temperatures(300, 100, 500.0, 250.0, QUANTUM)
run = equilibrium.simulate_energy_exchange(state, 8 * state.total_quanta, rng)
ledger = equilibrium.entropy_produced(run) / K_B

total_q = state.total_quanta
q = np.arange(total_q + 1)
exact_count = (equilibrium.einstein_log_multiplicity(q, 300)
               + equilibrium.einstein_log_multiplicity(total_q - q, 100))
ceiling = exact_count.max() - exact_count[state.q_a]

fig, (temps, books) = plt.subplots(2, 1, figsize=(8, 5.5), sharex=True)
temps.plot(run.steps, run.temperature_a, color="#dc2626", lw=0.8, label="T_A")
temps.plot(run.steps, run.temperature_b, color="#2563eb", lw=0.8, label="T_B")
temps.set_ylabel("T (K)")
temps.legend()
books.plot(run.steps, ledger, color="black", lw=0.8)
books.axhline(ceiling, color="grey", ls="--")
books.set_xlabel("step")
books.set_ylabel("Delta S_tot / k_B")
plt.tight_layout()
plt.show()

steps_down = np.diff(ledger) < 0
drawdown = np.max(np.maximum.accumulate(ledger) - ledger)
print(f"final ledger value               = {ledger[-1]:.4f} k_B")
print(f"largest total entropy available  = {ceiling:.4f} k_B")
print(f"fraction of steps that went DOWN = {steps_down.mean():.3f}")
print(f"largest single-step decrease     = {-np.diff(ledger).min():.4f} k_B")
print(f"deepest fall below running best  = {drawdown:.4f} k_B")

כמעט חמישית מכל הצעדים מורידים את האנטרופיה הכוללת — בכל פעם שקוונט קופץ לכיוון ה"לא נכון".
כל ירידה זעירה, והמאזן בכל זאת מטפס יותר מפי מאה מכל מה שהוא נסוג אי פעם לאחור, ומתיישר מעט
מתחת לאנטרופיה הכוללת הגדולה ביותר שיש לחלוקה כלשהי של האנרגיה הזו. מונוטוני *בתוך הרעש* — זה
כל מה שסימולציה יכולה להראות. ההוכחה שהמישור העליון הוא המקום שבו הוא נעצר היא טיעון הספירה
של מודול 08, לא הריצה הזו.

הסתייגות כנה אחת: כלל הקפיצה של מודול 01 מזיז קוונטים כאילו היו מתויגים, ולכן הרעד שלו לטווח
ארוך סביב השיא צר יותר מזה של מוצק איינשטיין אמיתי. השיא שלו נמצא באותה חלוקה, וזה מה שהמאזן
בודק.

### שלושה מחירים למגע אחד

אפשר לתמחר את אותה רלקסציה בספירה המדויקת (המאזן), במשטח סטירלינג החלק, ובנוסחת הקלורימטריה
$C_A \ln(T_{\text{eq}}/T_{A,0}) + C_B \ln(T_{\text{eq}}/T_{B,0})$
בעזרת קיבולי החום של מודול 01, $C = n \kB$.

In [ ]:
surface = (fundamental.einstein_solid_entropy(q * QUANTUM, 300, QUANTUM)
           + fundamental.einstein_solid_entropy((total_q - q) * QUANTUM, 100, QUANTUM)) / K_B
closed = fundamental.contact_entropy_production(
    state.heat_capacity_a, state.temperature_a, state.heat_capacity_b, state.temperature_b) / K_B
print(f"exact count (ledger ceiling) = {ceiling:.3f} k_B")
print(f"Stirling surface             = {surface.max() - surface[state.q_a]:.3f} k_B")
print(f"calorimetry formula          = {closed:.3f} k_B")

print("\nshrink the quantum: the Stirling/calorimetry gap is the high-temperature approximation")
for quantum_in_kb in (10.0, 5.0, 2.5, 1.25):
    eps = quantum_in_kb * K_B
    st = equilibrium.from_temperatures(300, 100, 500.0, 250.0, eps)
    qq = np.arange(st.total_quanta + 1)
    surf = (fundamental.einstein_solid_entropy(qq * eps, 300, eps)
            + fundamental.einstein_solid_entropy((st.total_quanta - qq) * eps, 100, eps)) / K_B
    formula = fundamental.contact_entropy_production(
        st.heat_capacity_a, st.temperature_a, st.heat_capacity_b, st.temperature_b) / K_B
    print(f"  epsilon/k_B = {quantum_in_kb:5.2f} K:  surface {surf.max() - surf[st.q_a]:.4f}, "
          f"formula {formula:.4f},  gap {formula - (surf.max() - surf[st.q_a]):.4f}")

## חלק 6 — האם המאזן בלתי תלוי בזרע?

ריצה אחת היא אנקדוטה. חזרו עליה עם זרעים בלתי תלויים והשוו את הפיזור של הערך הסופי לגודלו.

In [ ]:
def ledger_final(generator):
    trial = equilibrium.simulate_energy_exchange(state, 8 * state.total_quanta, generator)
    return float(equilibrium.entropy_produced(trial)[-1] / K_B)


def worst_step(generator):
    trial = equilibrium.simulate_energy_exchange(state, 8 * state.total_quanta, generator)
    return float(-np.diff(equilibrium.entropy_produced(trial)).min() / K_B)


finals = seed_study(ledger_final, n_seeds=8)
dips = seed_study(worst_step, n_seeds=8)
print(f"final value:          {finals.mean:.4f} +/- {finals.standard_error:.4f} k_B   "
      f"(ceiling {ceiling:.4f})")
print(f"worst single step:    {dips.mean:.5f} +/- {dips.standard_error:.5f} k_B")

## חלק 7 — אקסטנסיביות, בשלוש דרכים

**התפשטות חופשית.** הכפילו את הנפח של 10 000 אטומי ארגון באנרגיה קבועה. חשבו את $\Delta S$
מתוך משטח סאקור–טטרודה, מתוך נוסחת הגז האידיאלי של מודול 07, ומתוך החום שנקלט באיזותרמה הפיכה
בין אותם שני מצבים, חלקי הטמפרטורה שלה.

**אוילר וגיבס–דוהם.** בדקו את שניהם על שני היחסים האקסטנסיביים, ועל ה"כוכב" המוחזק בכבידה של
עצמו, שהאנטרופיה שלו אינה אקסטנסיבית.

In [ ]:
n_small, v1, temperature = 10_000, 1e-20, 300.0
u_small = 1.5 * n_small * K_B * temperature
from_surface = float(argon(u_small, 2 * v1, n_small) - argon(u_small, v1, n_small))
gas = processes.EquilibriumState.from_temperature(n_small, temperature, v1)
from_formula = engines.entropy_change_of_gas(processes.free_expansion(gas, 2 * v1))
from_isotherm = processes.isothermal(gas, 2 * v1).heat / temperature
print(f"N k_B ln 2        = {n_small * K_B * np.log(2):.6e} J/K")
print(f"Sackur-Tetrode    = {from_surface:.6e} J/K")
print(f"module 07 formula = {from_formula:.6e} J/K")
print(f"Q_rev / T         = {from_isotherm:.6e} J/K")

# Flow versus production: the same entropy change reached two ways.
bath = -processes.isothermal(gas, 2 * v1).heat / temperature  # the bath gave that heat up
print("\n                       gas          surroundings   produced   (J/K)")
print(f"reversible isotherm  {from_isotherm:+.3e}   {bath:+.3e}     {from_isotherm + bath:+.1e}")
print(f"free expansion       {from_surface:+.3e}   {0.0:+.3e}     {from_surface:+.3e}")

star = fundamental.self_gravitating_gas(1e-20)
print("\nEuler residual |U - TS + PV - mu N| / |U|:")
print(f"  argon          {fundamental.euler_residual(argon, u, v, n_atoms):.2e}")
print(f"  Einstein solid {fundamental.euler_residual(solid, 20 * 300 * QUANTUM, 1.0, 300.0):.2e}")
print(f"  star           {fundamental.euler_residual(star, -1e-16, 1.0, 1000.0):.6f}")

print("\nGibbs-Duhem residual along a displacement of size d (argon):")
for d in (4e-2, 2e-2, 1e-2, 5e-3):
    r = fundamental.gibbs_duhem_residual(argon, u, v, n_atoms, d * u, 2 * d * v, -d * n_atoms)
    print(f"  d = {d:.4f}:  {r:.3e}")

שלוש הדרכים אל ההתפשטות החופשית מסכימות: האנטרופיה עלתה ב-$N \kB \ln 2$ אף שלא זרם פנימה
חום, והמסלול ההפיך היה רק מכשיר לחישובה (ניבוי 3). הטבלה הקטנה מראה את החצי השני של הסיפור.
לאורך האיזותרמה ההפיכה הגז מרוויח בדיוק את מה שהאמבט מפסיד: האנטרופיה *זורמת*, ושום
אנטרופיה אינה מיוצרת. בהתפשטות החופשית הגז מרוויח את אותה כמות ושום דבר בשום מקום אינו
מפסיד אותה: כולה מיוצרת. יחס אוילר מתאזן עבור שני היחסים
האקסטנסיביים ונכשל עבור הכוכב בדיוק ב-$2|U|$, וזה מה שאנטרופיה לא-אקסטנסיבית מנבאת — הבדיקה
יכולה להיכשל, וזה מה שהופך אותה לבדיקה. השארית של גיבס–דוהם קטנה פי ארבעה בכל פעם שההעתק
מוקטן לחצי.

## חלק 8 — יציבות היא עקמומיות

קיבול החום הוא מינוס אחד חלקי המכפלה של $T^2$ בעקמומיות של $S(U)$. קראו אותו, ואז הביאו שני
כוכבים למגע וחפשו את השיא של האנטרופיה הכוללת שלהם.

In [ ]:
c_argon = fundamental.heat_capacity_of(argon, u, v, n_atoms) / (n_atoms * K_B)
c_star = fundamental.heat_capacity_of(star, -1e-16, 1.0, 1000.0) / (1000 * K_B)
print(f"argon  C_V / (N k_B) = {c_argon:.5f}"
      f"   stable: {fundamental.is_stable(argon, u, v, n_atoms)}")
print(f"star   C   / (N k_B) = {c_star:.5f}"
      f"   stable: {fundamental.is_stable(star, -1e-16, 1.0, 1000.0)}")

stars = fundamental.Composite(star, star, fundamental.Part(-1e-16, 1.0, 1000.0),
                              fundamental.Part(-1e-16, 1.0, 1000.0))
u_star, s1, s2 = stars.energy_scan(n_points=401, margin=0.05)
two_solids = fundamental.Composite(solid, solid, fundamental.Part(1e-19, 1.0, 300.0),
                                   fundamental.Part(1e-19, 1.0, 300.0))
u_sol, t1, t2 = two_solids.energy_scan(n_points=401, margin=0.05)

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.5))
left.plot(u_sol / two_solids.total("energy"), (t1 + t2 - (t1 + t2).max()) / K_B, color="black")
left.set_title("two identical solids")
right.plot(u_star / stars.total("energy"), (s1 + s2 - (s1 + s2).max()) / K_B, color="#dc2626")
right.set_title("two identical stars")
for ax in (left, right):
    ax.axvline(0.5, color="grey", ls=":")
    ax.set_xlabel("share of the total energy on side A")
    ax.set_ylabel("(S_tot - S_max) / k_B")
plt.tight_layout()
plt.show()

לשני מוצקים זהים יש מקסימום בחלוקה השווה: כל פלוקטואציה מורידה את האנטרופיה הכוללת ומתבטלת.
לשני כוכבים זהים יש שם *מינימום*: כל פלוקטואציה מעלה את האנטרופיה הכוללת, והצמד בורח לעבר
הקצוות, כשכוכב אחד נוטל את האנרגיה (ניבוי 4). קיבול חום שלילי ואנטרופיה קמורה הם אותה טענה.

## חלק 9 — מדידת ייצור אנטרופיה בניסוי ממשי

הקובץ `data/09-mixing-calorimetry.csv` הוא ריצת ערבוב: $0.150\ \mathrm{kg}$ של מים חמים נמזגים
לתוך $0.200\ \mathrm{kg}$ של מים קרים בכוס קלקר (קראו את הכותרת של הקובץ כדי לדעת בדיוק מה הוא
מכיל). אף קריאה בודדת אינה הטמפרטורה הסופית — התערובת ממשיכה לאבד חום לחדר — ולכן התאימו
עקומה לכל סדרת מדידות ובצעו אקסטרפולציה אל רגע הערבוב, $t = 0$.

In [ ]:
from pathlib import Path

try:
    import piplite  # noqa: F401
except ImportError:
    # Desktop / nbmake: the repository root is three directories up from this notebook.
    csv_path = Path("..", "..", "..", "data", "09-mixing-calorimetry.csv")
else:
    # JupyterLite bundles only the notebooks/ tree (jupyter_lite_config.json's
    # LiteBuildConfig.contents), so the browser gets its own copy of the CSV co-located here.
    csv_path = Path("data", "09-mixing-calorimetry.csv")

log = np.genfromtxt(csv_path, delimiter=",", comments="#")
t, t_hot, t_cold, t_mix = log.T
before, after = ~np.isnan(t_hot), (~np.isnan(t_mix)) & (t >= 60)


def extrapolate_to_zero(times, temps):
    # Straight-line fit; the intercept is the temperature at t = 0, with its standard error.
    coeffs, cov = np.polyfit(times, temps, 1, cov=True)
    return coeffs[1], float(np.sqrt(cov[1, 1]))


hot0, d_hot = extrapolate_to_zero(t[before], t_hot[before])
cold0, d_cold = extrapolate_to_zero(t[before], t_cold[before])
final, d_final = extrapolate_to_zero(t[after], t_mix[after])

c_water = 4184.0
c_hot, c_cold = 0.150 * c_water, 0.200 * c_water
predicted_final = (c_hot * hot0 + c_cold * cold0) / (c_hot + c_cold)


def cup_capacity(t_h, t_c, t_f):
    # Energy balance: what the hot water gave up and the cold water did not receive went into
    # the cup and the probe, which started at the cold water's temperature.
    return (c_hot * (t_h - t_f) - c_cold * (t_f - t_c)) / (t_f - t_c)


def produced(t_h, t_c, t_f):
    # Every body that exchanged heat: each changes by C ln(T_final / T_start), module 07's result.
    return (c_hot * np.log(t_f / t_h)
            + (c_cold + cup_capacity(t_h, t_c, t_f)) * np.log(t_f / t_c))


value = produced(hot0, cold0, final)
error = np.sqrt(sum(
    ((produced(hot0 + dh, cold0 + dc, final + df) - value)) ** 2
    for dh, dc, df in ((d_hot, 0, 0), (0, d_cold, 0), (0, 0, d_final))
))

plt.figure(figsize=(8, 3.8))
plt.plot(t[before], t_hot[before], ".", color="#dc2626", label="hot vessel")
plt.plot(t[before], t_cold[before], ".", color="#2563eb", label="cold vessel")
plt.plot(t[~np.isnan(t_mix)], t_mix[~np.isnan(t_mix)], ".", color="black", label="mixture")
plt.axhline(final, color="grey", ls="--", lw=0.8)
plt.axvline(0, color="grey", lw=0.8)
plt.xlabel("t (s)")
plt.ylabel("T (K)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"at t = 0:  hot {hot0:.2f} +/- {d_hot:.2f} K,  cold {cold0:.2f} +/- {d_cold:.2f} K")
print(f"final temperature, extrapolated: {final:.2f} +/- {d_final:.2f} K")
print(f"final temperature, if only the two waters shared the heat: {predicted_final:.2f} K")
print(f"cup and probe, from the energy balance: {cup_capacity(hot0, cold0, final):.1f} J/K")
print(f"\nmeasurement: Delta S_total = {value:.3f} +/- {error:.3f} J/K")
print(f"leaving the cup out of the books:   "
      f"{c_hot * np.log(final / hot0) + c_cold * np.log(final / cold0):.2f} J/K")
print(f"prediction for two waters alone:    "
      f"{fundamental.contact_entropy_production(c_hot, hot0, c_cold, cold0):.2f} J/K")

האנטרופיה המיוצרת חיובית ונמצאת הרחק מחוץ לתחום השגיאה שלה, כפי שהיא חייבת להיות. הטמפרטורה
הסופית הנמדדת נמצאת כ-$0.4\ \mathrm{K}$ מתחת למה ששתי כמויות המים לבדן היו מגיעות אליו: הכוס
ומד-החום נטלו חלק מן החום. מאזן האנרגיה אומר כמה. קלורימטריסט קורא לזה *שווה-ערך המים* של
הכוס, והוא יוצא כאן כ-$26\ \mathrm{J\,K^{-1}}$, בערך שישה גרם מים.

עכשיו התבוננו בשורה שמשאירה את הכוס בחוץ. היא שגויה מאוד, והסיבה ראויה לזכירה: המים החמים
מסרו יותר חום משהמים הקרים קיבלו, ולכן ספרי חשבונות המונים רק את שתי כמויות המים אינם משמרים
אנרגיה, ומאזן אנטרופיה הבנוי עליהם חסר משמעות. ייצור אנטרופיה הוא סכום על פני **כל** גוף
שהחליף חום. הכללת הכוס מחזירה את המדידה להתאמה עם הניבוי עבור שתי כמויות המים.

## חלק 10 — בדיקות אוטומטיות

סימולציה שלא בדקתם היא תמונה, לא ראיה. אלה אותן בדיקות שרצות בערכת המבחנים של הפרויקט.

In [ ]:
# 1. The slopes of Sackur-Tetrode are the ideal-gas equations of state.
assert relative_error(slopes.temperature, 300.0) < 1e-7
assert relative_error(slopes.pressure, 1e5) < 1e-7

# 2. The maximiser lands on equal temperatures and the 1 : 10 energy split (Part 2).
assert relative_error(best.slopes_a().temperature, best.slopes_b().temperature) < 1e-6
assert relative_error(best.b.energy / best.a.energy, 10.0) < 1e-6

# 3. A freed piston equalises the pressures; perforating afterwards changes nothing (Part 3).
assert relative_error(piston.slopes_a().pressure, piston.slopes_b().pressure) < 1e-6
assert relative_error(piston.released("energy", "particles").total_entropy,
                      piston.total_entropy) < 1e-12

# 4. The central difference is second order (Part 4).
assert 1.9 < study.observed_order < 2.1

# 5. The ledger dips on single steps yet ends at the ceiling (Part 5).
assert steps_down.any()
assert ledger[-1] <= ceiling and relative_error(ledger[-1], ceiling) < 0.01

# 6. Free expansion is N k_B ln 2 by all three routes (Part 7).
for route in (from_surface, from_formula, from_isotherm):
    assert relative_error(route, n_small * K_B * np.log(2)) < 1e-9

# 6b. The reversible isotherm moves entropy without producing any (Part 7).
assert abs(from_isotherm + bath) < 1e-9 * from_isotherm

# 7. Euler holds for extensive relations and fails for the star (Part 7).
assert fundamental.euler_residual(argon, u, v, n_atoms) < 1e-6
assert relative_error(fundamental.euler_residual(star, -1e-16, 1.0, 1000.0), 2.0) < 1e-6

# 8. Mixing produces entropy, beyond the measurement error (Part 9).
assert value - 3 * error > 0

print("all checks passed")

## חלק 11 — לחקור בעצמכם

בחרו את הגדלים של שני המוצקים, את המספר הכולל של הקוונטים ואת המקום שבו האנרגיה מתחילה, ואז
לחצו על **Run Interact**. הלוח השמאלי מציג את האנטרופיה הכוללת לאורך החלוקה, עם נקודת ההתחלה
והשיא; הלוח הימני מריץ את הסימולציה של מודול 01 מנקודת ההתחלה הזו ומנהל את המאזן. שלושה דברים
שכדאי לנסות:

1. עשו את המוצקים שווים, ואז עשו אחד מהם גדול פי מאה מן השני. היכן השיא?
2. התחילו *בשיא*. מה המאזן עושה?
3. כווצו את שני המוצקים לקומץ מתנדים והתבוננו ברעש של המאזן גדל.

In [ ]:
import ipywidgets as widgets


def explore(n_a=40, n_b=400, quanta=2000, start_share=0.8):
    q_start = int(round(start_share * quanta))
    pair_now = fundamental.Composite(
        solid, solid,
        fundamental.Part(max(q_start, 1) * QUANTUM, 1.0, float(n_a)),
        fundamental.Part(max(quanta - q_start, 1) * QUANTUM, 1.0, float(n_b)))
    ua, sa, sb = pair_now.energy_scan(n_points=801, margin=0.002)
    total_now = pair_now.total("energy")
    peak_now = pair_now.released("energy")

    state_now = equilibrium.TwoBodyState(n_a=n_a, n_b=n_b, q_a=q_start,
                                         q_b=quanta - q_start, quantum=QUANTUM)
    trial = equilibrium.simulate_energy_exchange(state_now, 6 * quanta, rng)
    books_now = equilibrium.entropy_produced(trial) / K_B

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
    left.plot(ua / total_now, (sa + sb - (sa + sb).max()) / K_B, color="black")
    left.axvline(start_share, color="#dc2626", ls="--", label="start")
    left.axvline(peak_now.a.energy / total_now, color="grey", ls=":", label="peak")
    left.set_ylim(max(float(((sa + sb) - (sa + sb).max()).min() / K_B), -200), 5)
    left.set_xlabel("U_A / U")
    left.set_ylabel("(S_tot - S_max) / k_B")
    left.legend()
    right.plot(trial.steps, books_now, color="black", lw=0.8)
    right.set_xlabel("step")
    right.set_ylabel("Delta S_tot / k_B")
    plt.tight_layout()
    plt.show()
    print(f"peak at U_A/U = {peak_now.a.energy / total_now:.4f}   "
          f"T_A = T_B = {peak_now.slopes_a().temperature:.2f} K   "
          f"ledger final = {books_now[-1]:.2f} k_B")


widgets.interact_manual(
    explore,
    n_a=widgets.IntSlider(min=16, max=4096, step=8, value=40, description="n_A"),
    n_b=widgets.IntSlider(min=16, max=4096, step=8, value=400, description="n_B"),
    quanta=widgets.IntSlider(min=100, max=20000, step=100, value=2000, description="quanta"),
    start_share=widgets.FloatSlider(min=0.02, max=0.98, step=0.02, value=0.8,
                                    description="start U_A/U"),
);

## בחנו את הבנתכם

הריצו את התא שלהלן לחידון הנבדק אוטומטית. אותן שאלות, עם הסברים כתובים לכל אפשרות, נמצאות
בדף המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "09-fundamental-relation.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

רשמו כמה משפטים על כל אחת, בתא שלהלן.

1. מה ניבאתם שהתברר כשגוי, ומה בדיוק היה הפגם בנימוקכם?
2. הממקסם בחלק 2 מעולם לא השתמש במילה "טמפרטורה", ובכל זאת נחת על טמפרטורות שוות. הסבירו
   מדוע זה היה בלתי נמנע.
3. המאזן בחלק 5 ירד באלפי צעדים. אמרו במדויק מה החוק השני טוען ומה אינו טוען על צעד בודד.
4. השארית של אוילר עבור הכוכב הייתה בדיוק 2, ביחידות של גודל האנרגיה שלו: יחס אוילר
   החמיץ פעמיים את האנרגיה. איזו תכונה של האנטרופיה שלו המספר הזה מודד?

**התשובות שלכם:**

1.
2.
3.
4.